In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [8]:
final_df = pd.read_csv('all_points_combined.csv')

# Вычисляем фактор коррекции сразу для всего столбца (только там, где влажность > 40)
# Если влажность <= 40, фактор будет равен 1.0 (то есть значения не изменятся)
correction_factor = np.where(
    final_df['Humidity'] > 40,
    1 + 0.25 * ((final_df['Humidity'] / 100) ** 3),
    1.0
)

# Корректируем все три столбца одной быстрой операцией
final_df['PM1.0'] = final_df['PM1.0'] / correction_factor
final_df['PM2.5'] = final_df['PM2.5'] / correction_factor
final_df['PM10'] = final_df['PM10'] / correction_factor

# Группируем по имени исходного файла (каждый файл — это одна точка)
points_summary = final_df.groupby('source_file').agg({
    'PM1.0': 'median',   # Используем медиану, чтобы отсечь случайный шум
    'PM2.5': 'median',
    'PM10': 'median',
    'Humidity': 'mean', # Для климата можно взять среднее арифметическое
    'Temperature': 'mean'
}).reset_index()

# Теперь к этой таблице нужно добавить координаты (Широту и Долготу) для каждого файла
# Создайте словарь с координатами ваших 15 точек:
coordinates = {
    'Gagarinskiy_magistral.CSV': [55.703604, 37.573724], 
    'Gagarinskiy_park.CSV': [55.702834, 37.556855],
    'Gagarinskiy_dvor.CSV': [55.692895, 37.546213],
    'Lomonosovskiy_park.CSV': [55.684228, 37.530345],
    'Lomonosovskiy_magistral.CSV': [55.683273, 37.537094],
    'Lomonosovskiy_dvor.CSV': [55.681823, 37.54301],
    'Akademicheskaya_park.CSV':[55.692126, 37.569751],
    'Akademicheskaya_magistal.CSV':[55.68527, 37.571208],
    'Akademicheskaya_dvor.CSV':[55.687147, 37.578238],
    'Akademicheskaya_station.CSV':[55.680032, 37.58385],
    'Kotlovka_dvor.CSV':[55.678995, 37.598741],
    'Kotlovka_magistral.CSV':[55.677883, 37.602037],
    'Kotlovka_park.CSV':[55.669676, 37.596294],
    'Zuzino_dvor.CSV':[55.655532, 37.591305],
    'Zuzino_magistral.CSV':[55.651856, 37.592599],
    'Zuzino_park.CSV':[55.650246, 37.585329],
    'Cheremushki_dvor.CSV':[55.660617, 37.5673],
    'Cheremushki_magistral.CSV':[55.660373, 37.560804],
    'Cheremushki_park.CSV':[55.660021, 37.556267],
    'Obruchevskiy_magistral.CSV':[55.658074, 37.519931],
    'Obruchevskiy_dvor.CSV':[55.662574, 37.513962],
    'Obruchevskiy_park.CSV':[55.656181, 37.508796],
    'Konkovo_dvor.CSV':[55.653024, 37.520745],
    'Konkovo_magistral.CSV':[55.651194, 37.534587],
    'Konkovo_station.CSV':[55.649728, 37.536931],
    'Konkovo_park.CSV':[55.641384, 37.516494],
    'Tepliy_dvor.CSV':[55.629923, 37.511689],
    'Tepliy_park.CSV':[55.63063, 37.49633],
    'Tepliy_magistral.CSV':[55.617753, 37.502494],
    'Yasenevo_park.CSV':[55.617114, 37.522163],
    'Yasenevo_dvor.CSV':[55.606637, 37.526421],
    'Yasenevo_magistral.CSV':[55.604983, 37.539005],
    'NButovo_dvor.CSV':[55.572064, 37.569188],
    'NButovo_magistral.CSV':[55.569004, 37.576292],
    'NButovo_park.CSV':[55.561981, 37.565758],
    'SButovo_dvor.CSV':[55.548915, 37.545038],
    'SButovo_magistral.CSV':[55.543107, 37.549315],
    'SButovo_park.CSV':[55.540242, 37.527855]
}

# Добавляем координаты в итоговую таблицу точек
points_summary['latitude'] = points_summary['source_file'].map(lambda x: coordinates.get(x, [None, None])[0])
points_summary['longitude'] = points_summary['source_file'].map(lambda x: coordinates.get(x, [None, None])[1])

# Сохраняем результат для первой карты
points_summary.to_csv('map_1_points_data.csv', index=False)
print("Медиана PM2.5 ПОСЛЕ коррекции:", final_df['PM2.5'].median())

# Чтобы узнать, какими данные были ДО, можно просто заново прочитать файл на секунду:
original_df = pd.read_csv('all_points_combined.csv')
print("Медиана PM2.5 ДО коррекции:   ", original_df['PM2.5'].median())
print("Средняя влажность в датасете:  ", original_df['Humidity'].mean())

Медиана PM2.5 ПОСЛЕ коррекции: 18.506500203397003
Медиана PM2.5 ДО коррекции:    19.0
Средняя влажность в датасете:   53.82978779840849
